# 00 Prepare Data

            Objective: load the NICE/PROMISE-derived requirements dataset, create a transparent seed review table, and keep exactly 120 accepted seed capabilities for the main experiment.

            The automatic filter is intentionally conservative. Manual review remains part of the protocol: inspect `data/processed/seeds_review.csv`, edit `include`, `exclusion_reason`, and `capability_text_final` if needed, then rerun the validation cells.


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)

PROJECT_ROOT, CONFIG_PATH


## Load or Download NICE


In [ ]:
nice_path = PROJECT_ROOT / CONFIG["datasets"]["nice_local_path"]
            nice_url = CONFIG["datasets"]["nice_url"]

            if not nice_path.exists():
                print(f"NICE CSV not found at {nice_path}. Downloading from Zenodo...")
                eu.download_file(nice_url, nice_path, timeout_s=CONFIG["llm"]["timeout_s"])
            else:
                print(f"Using existing NICE CSV: {nice_path}")

            rows = eu.read_csv_rows(nice_path)
            text_column = eu.find_requirement_text_column(rows)
            print(f"Loaded {len(rows)} rows. Requirement text column: {text_column}")
            print(rows[0][text_column][:240])


## Build Review Table


In [ ]:
target_count = int(CONFIG["project"]["target_seed_count"])
            candidates = eu.make_seed_candidates(rows, target_count=target_count)

            review_fields = [
                "seed_id",
                "source_dataset",
                "original_requirement",
                "capability_text_auto",
                "auto_include",
                "auto_exclusion_reason",
                "include",
                "exclusion_reason",
                "capability_text_final",
            ]
            review_path = PROJECT_ROOT / "data/processed/seeds_review.csv"
            eu.write_csv_rows(review_path, candidates, fieldnames=review_fields)

            auto_ok = sum(1 for row in candidates if row["auto_include"] == "yes")
            selected = eu.load_reviewed_seeds(review_path, target_count=target_count, strict=False)
            print(f"Wrote review table: {review_path}")
            print(f"Automatic candidates passing filters: {auto_ok}")
            print(f"Currently included seeds: {len(selected)} / {target_count}")


## Validate Reviewed Seeds


In [ ]:
selected = eu.load_reviewed_seeds(review_path, target_count=target_count, strict=False)
            selected_path = PROJECT_ROOT / "data/processed/seeds_selected.csv"

            if len(selected) == target_count:
                eu.write_csv_rows(selected_path, selected)
                print(f"OK: exactly {target_count} included seeds.")
                print(f"Wrote selected seeds: {selected_path}")
            else:
                print(f"Review needed: found {len(selected)} included seeds, expected {target_count}.")
                print("Edit data/processed/seeds_review.csv, then rerun this cell.")

            selected[:3]
